# 01 - Entendimento dos Dados

Objetivo deste notebook: conhecer as tabelas brutas do dataset da Olist antes de fazer qualquer limpeza ou análise de negócio.

Nesta etapa vamos verificar:

- quais arquivos existem;
- quantas linhas e colunas cada tabela possui;
- quais sao as colunas principais;
- quais campos possuem valores ausentes;
- quais relacoes existem entre as tabelas.

## 1. Importar bibliotecas

Usaremos `pandas` para ler e explorar os arquivos CSV.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## 2. Definir caminhos

Este notebook esta dentro da pasta `notebooks`, entao usamos `..` para voltar para a raiz do projeto.

In [2]:
PROJECT_ROOT = Path("..").resolve()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

RAW_DATA_DIR

WindowsPath('C:/Users/lucas/brazilian-ecommerce-analytics/data/raw')

## 3. Listar arquivos CSV

Antes de analisar, confirmamos se todos os arquivos esperados estao na pasta correta.

In [3]:
csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

for file in csv_files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


## 4. Carregar tabelas

Vamos carregar cada CSV em um dicionario de DataFrames. O nome da tabela sera o nome do arquivo sem `.csv`.

In [4]:
tables = {
    file.stem: pd.read_csv(file)
    for file in csv_files
}

list(tables.keys())

['olist_customers_dataset',
 'olist_geolocation_dataset',
 'olist_order_items_dataset',
 'olist_order_payments_dataset',
 'olist_order_reviews_dataset',
 'olist_orders_dataset',
 'olist_products_dataset',
 'olist_sellers_dataset',
 'product_category_name_translation']

## 5. Resumo das tabelas

Aqui criamos uma visao geral com linhas, colunas e quantidade total de valores ausentes.

In [5]:
summary = []

for name, df in tables.items():
    summary.append({
        "table": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
    })

summary_df = pd.DataFrame(summary).sort_values("rows", ascending=False)
summary_df

,table,rows,columns,missing_values
1,olist_geolocation_dataset,1000163,5,0
2,olist_order_items_dataset,112650,7,0
3,olist_order_payments_dataset,103886,5,0
0,olist_customers_dataset,99441,5,0
5,olist_orders_dataset,99441,8,4908
4,olist_order_reviews_dataset,99224,7,145903
6,olist_products_dataset,32951,9,2448
7,olist_sellers_dataset,3095,4,0
8,product_category_name_translation,71,2,0


## 6. Colunas de cada tabela

Esta celula ajuda a entender o papel de cada arquivo.

In [6]:
for name, df in tables.items():
    print(f"\n{name}")
    print("-" * len(name))
    for column in df.columns:
        print(column)


olist_customers_dataset
-----------------------
customer_id
customer_unique_id
customer_zip_code_prefix
customer_city
customer_state

olist_geolocation_dataset
-------------------------
geolocation_zip_code_prefix
geolocation_lat
geolocation_lng
geolocation_city
geolocation_state

olist_order_items_dataset
-------------------------
order_id
order_item_id
product_id
seller_id
shipping_limit_date
price
freight_value

olist_order_payments_dataset
----------------------------
order_id
payment_sequential
payment_type
payment_installments
payment_value

olist_order_reviews_dataset
---------------------------
review_id
order_id
review_score
review_comment_title
review_comment_message
review_creation_date
review_answer_timestamp

olist_orders_dataset
--------------------
order_id
customer_id
order_status
order_purchase_timestamp
order_approved_at
order_delivered_carrier_date
order_delivered_customer_date
order_estimated_delivery_date

olist_products_dataset
----------------------
product_id
p

## 7. Valores ausentes por tabela

Valores ausentes nao sao necessariamente erro. Em entregas, por exemplo, datas vazias podem indicar pedidos cancelados ou ainda nao entregues.

In [7]:
for name, df in tables.items():
    missing = df.isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    print(f"\n{name}")
    print("-" * len(name))
    if missing.empty:
        print("Sem valores ausentes")
    else:
        print(missing)


olist_customers_dataset
-----------------------
Sem valores ausentes

olist_geolocation_dataset
-------------------------
Sem valores ausentes

olist_order_items_dataset
-------------------------
Sem valores ausentes

olist_order_payments_dataset
----------------------------
Sem valores ausentes

olist_order_reviews_dataset
---------------------------
review_comment_title      87656
review_comment_message    58247
dtype: int64

olist_orders_dataset
--------------------
order_delivered_customer_date    2965
order_delivered_carrier_date     1783
order_approved_at                 160
dtype: int64

olist_products_dataset
----------------------
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

olist_sellers_dataset
---------------------
Sem valores ausentes


## 8. Primeiras linhas das tabelas principais

Vamos visualizar exemplos das tabelas mais importantes para o projeto.

In [8]:
tables["olist_orders_dataset"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [9]:
tables["olist_order_items_dataset"].head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [10]:
tables["olist_order_reviews_dataset"].head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


## 9. Proximas perguntas

Depois desta primeira leitura, as proximas analises serao:

1. Converter colunas de data para o tipo correto.
2. Criar uma tabela analitica juntando pedidos, itens, produtos, clientes, vendedores e avaliacoes.
3. Calcular KPIs: receita, pedidos, ticket medio, frete medio, prazo de entrega e nota media.
4. Investigar a relacao entre atraso na entrega e avaliacao do cliente.